# Notebook 09 — Regime-Adaptive Multi-Scale Quantum ML

## What This Notebook Implements

This notebook implements the **minimal viable version** of the new algorithm proposed in
`QUANTUM_ML_ALGORITHM_RESEARCH_REPORT.md`:

## **RAM-QRSRE**
**Regime-Adaptive Multi-Scale Quantum Reservoir with Re-uploading Experts**

### Design goals

1. Preserve what already worked best in this repo:
   - **fixed quantum reservoirs** for wildfire classification
   - **temporal structure** for premium prediction
2. Avoid the failure modes seen in:
   - deep variational circuits
   - global fidelity-style kernels
3. Add a small trainable quantum head without letting optimization dominate the whole model

### Minimal viable implementation in this notebook

- **Three shallow fixed quantum reservoirs**
  - short-scale
  - seasonal / pooled-scale
  - trend-scale
- **Projected local observables**
  - single-qubit `Z`
  - single-qubit `X`
  - nearest-neighbor `ZZ`
- **A shallow data re-uploading quantum head**
  - trained layerwise: 1 layer → 2 layers
- **Two task-specific instantiations**
  - Task 1A wildfire day classification
  - Task 2 insurance premium forecasting

### Important note

This notebook is designed to be:

- more novel than standard VQC / kernel baselines,
- closer to current QML trainability guidance,
- and still light enough to run on a laptop.

It is **not** a claim of guaranteed quantum advantage.


---
## 0 — Imports & Configuration

In [ ]:
import json as _json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pennylane as qml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    classification_report,
    f1_score,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 150, "figure.figsize": (12, 5)})

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

RESULT_PREFIX = 'ram_qrsre'
CLOUD_MODE = False

N_RES_QUBITS = 4
RESERVOIR_OBS_PER_BANK = 3 * N_RES_QUBITS  # Z + X + NN-ZZ
HEAD_LAYER_SCHEDULE = (1, 2)
N_REGIMES = 3

TASK1_TRAIN_CAP = None if CLOUD_MODE else 300
TASK1_TEST_CAP = None if CLOUD_MODE else 200
TASK2_TRAIN_CAP = None if CLOUD_MODE else 160
TASK2_TEST_CAP = None if CLOUD_MODE else 200

print(f"PennyLane {qml.__version__} | Torch {torch.__version__}")
print(f"Reservoir qubits: {N_RES_QUBITS}")
print(f"Layerwise schedule: {HEAD_LAYER_SCHEDULE}")
print(f"Regime experts: {N_REGIMES}")

try:
    IN_NOTEBOOK = get_ipython() is not None
except NameError:
    IN_NOTEBOOK = False


---
## 1 — Task 1A: Load Wildfire Data and Build Multiscale Views

In [ ]:
wf = pd.read_csv("../data/wildfire_weather_daily.csv")
wf["DATE"] = pd.to_datetime(wf["DATE"], errors="coerce")
wf = wf.sort_values("DATE").reset_index(drop=True)

base_cols = [
    "PRECIPITATION",
    "MAX_TEMP",
    "MIN_TEMP",
    "AVG_WIND_SPEED",
    "TEMP_RANGE",
    "WIND_TEMP_RATIO",
    "LAGGED_PRECIPITATION",
    "LAGGED_AVG_WIND_SPEED",
]

for col in ["PRECIPITATION", "MAX_TEMP", "MIN_TEMP", "AVG_WIND_SPEED", "TEMP_RANGE", "WIND_TEMP_RATIO"]:
    wf[f"{col}_7D"] = wf[col].shift(1).rolling(7, min_periods=3).mean()
    wf[f"{col}_30D"] = wf[col].shift(1).rolling(30, min_periods=10).mean()

wf["FIRE_RATE_30D"] = wf["FIRE_START_DAY"].astype(float).shift(1).rolling(30, min_periods=10).mean()
wf["FIRE_RATE_90D"] = wf["FIRE_START_DAY"].astype(float).shift(1).rolling(90, min_periods=20).mean()
wf["MONTH_SIN"] = np.sin(2 * np.pi * wf["MONTH"] / 12.0)
wf["MONTH_COS"] = np.cos(2 * np.pi * wf["MONTH"] / 12.0)
wf["DOY_SIN"] = np.sin(2 * np.pi * wf["DAY_OF_YEAR"] / 365.0)
wf["DOY_COS"] = np.cos(2 * np.pi * wf["DAY_OF_YEAR"] / 365.0)
wf["DRYNESS_PROXY"] = wf["MAX_TEMP_30D"] / (1.0 + wf["PRECIPITATION_30D"].abs())
wf["WIND_STRESS_PROXY"] = wf["AVG_WIND_SPEED_30D"] * wf["TEMP_RANGE_30D"]

task1_short_cols = base_cols
task1_seasonal_cols = [
    "PRECIPITATION_7D",
    "MAX_TEMP_7D",
    "MIN_TEMP_7D",
    "AVG_WIND_SPEED_7D",
    "PRECIPITATION_30D",
    "MAX_TEMP_30D",
    "MIN_TEMP_30D",
    "AVG_WIND_SPEED_30D",
]
task1_trend_cols = [
    "MONTH_SIN",
    "MONTH_COS",
    "DOY_SIN",
    "DOY_COS",
    "FIRE_RATE_30D",
    "FIRE_RATE_90D",
    "DRYNESS_PROXY",
    "WIND_STRESS_PROXY",
]

task1_all_needed = task1_short_cols + task1_seasonal_cols + task1_trend_cols + ["FIRE_START_DAY", "YEAR"]
wf = wf.dropna(subset=task1_all_needed).copy()

train_mask_1a = wf["YEAR"] < 2021
test_mask_1a = wf["YEAR"] == 2021

X1_train_views_raw = {
    "short": wf.loc[train_mask_1a, task1_short_cols].to_numpy(dtype=float),
    "seasonal": wf.loc[train_mask_1a, task1_seasonal_cols].to_numpy(dtype=float),
    "trend": wf.loc[train_mask_1a, task1_trend_cols].to_numpy(dtype=float),
}
X1_test_views_raw = {
    "short": wf.loc[test_mask_1a, task1_short_cols].to_numpy(dtype=float),
    "seasonal": wf.loc[test_mask_1a, task1_seasonal_cols].to_numpy(dtype=float),
    "trend": wf.loc[test_mask_1a, task1_trend_cols].to_numpy(dtype=float),
}
y1_train = wf.loc[train_mask_1a, "FIRE_START_DAY"].astype(int).to_numpy()
y1_test = wf.loc[test_mask_1a, "FIRE_START_DAY"].astype(int).to_numpy()
task1_regime_signal_train = wf.loc[train_mask_1a, "DRYNESS_PROXY"].to_numpy(dtype=float)
task1_regime_signal_test = wf.loc[test_mask_1a, "DRYNESS_PROXY"].to_numpy(dtype=float)

if TASK1_TRAIN_CAP is not None:
    X1_train_views_raw = {k: v[:TASK1_TRAIN_CAP] for k, v in X1_train_views_raw.items()}
    y1_train = y1_train[:TASK1_TRAIN_CAP]
    task1_regime_signal_train = task1_regime_signal_train[:TASK1_TRAIN_CAP]
if TASK1_TEST_CAP is not None:
    X1_test_views_raw = {k: v[:TASK1_TEST_CAP] for k, v in X1_test_views_raw.items()}
    y1_test = y1_test[:TASK1_TEST_CAP]
    task1_regime_signal_test = task1_regime_signal_test[:TASK1_TEST_CAP]

print("Task 1A raw views:")
for k, v in X1_train_views_raw.items():
    print(f"  {k:<8s} train={v.shape} | test={X1_test_views_raw[k].shape}")
print(f"  positives train={y1_train.sum()}/{len(y1_train)} | test={y1_test.sum()}/{len(y1_test)}")


---
## 2 — Task 2: Load Insurance Data and Build Multi-Scale Sequence Views

In [ ]:
def find_col(df, keywords):
    kw = [k.lower() for k in keywords]
    for c in df.columns:
        norm = c.replace("\n", " ").strip().lower()
        if all(k in norm for k in kw):
            return c
    return None

def load_ho(path, years):
    frames = []
    for yr in years:
        df = pd.read_excel(path, sheet_name=f"{yr}HO", header=1, engine="openpyxl")
        df.columns = [c.replace("\n", " ").strip() for c in df.columns]
        df = df.rename(columns={df.columns[0]: "ZIP_Code"})
        df["ZIP_Code"] = pd.to_numeric(df["ZIP_Code"], errors="coerce")
        df = df.dropna(subset=["ZIP_Code"])
        df["ZIP_Code"] = df["ZIP_Code"].astype(int)
        df["Year"] = yr
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

ins_a = load_ho("../data/insurance_2018_2019.XLS", [2018, 2019])
ins_b = load_ho("../data/insurance_2020_2021.XLS", [2020, 2021])
common_cols = sorted(set(ins_a.columns) & set(ins_b.columns))
ins = pd.concat([ins_a[common_cols], ins_b[common_cols]], ignore_index=True)
ins = ins.sort_values(["ZIP_Code", "Year"]).reset_index(drop=True)

col_risk = find_col(ins, ["avg", "fire", "risk"])
col_prem = find_col(ins, ["earned", "premium"])
col_exp = find_col(ins, ["earned", "exposure"])
col_high = find_col(ins, ["high", "fire", "risk", "exposure"])
col_vhigh = find_col(ins, ["very", "high", "fire", "risk", "exposure"])

loss_cols = sorted(c for c in ins.columns if "fire" in c.lower() and "incurred losses" in c.lower())
smoke_cols = sorted(c for c in ins.columns if "smoke" in c.lower() and "incurred losses" in c.lower())

ins["total_fire_loss"] = ins[loss_cols].fillna(0).sum(axis=1)
ins["total_smoke_loss"] = ins[smoke_cols].fillna(0).sum(axis=1)
ins["total_insured_loss"] = ins["total_fire_loss"] + ins["total_smoke_loss"]
ins["loss_ratio"] = ins["total_insured_loss"] / ins[col_prem].replace(0, np.nan)
ins["prem_per_pol"] = ins[col_prem] / ins[col_exp].replace(0, np.nan)
ins["high_risk_frac"] = (
    (ins[col_high].fillna(0) + ins[col_vhigh].fillna(0)) / ins[col_exp].replace(0, np.nan)
)
ins["premium_growth"] = ins.groupby("ZIP_Code")[col_prem].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
ins["loss_growth"] = ins.groupby("ZIP_Code")["total_insured_loss"].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)

task2_feature_cols = [
    col_risk,
    col_exp,
    "total_insured_loss",
    "loss_ratio",
    "prem_per_pol",
    "high_risk_frac",
    "premium_growth",
    "loss_growth",
]

ins_ts = ins.dropna(subset=task2_feature_cols + [col_prem, "Year"]).copy()

short_views, seasonal_views, trend_views, y2, years2 = [], [], [], [], []
for _, grp in ins_ts.groupby("ZIP_Code"):
    grp = grp.sort_values("Year")
    feats = grp[task2_feature_cols].to_numpy(dtype=float)
    prems = grp[col_prem].to_numpy(dtype=float)
    yrs = grp["Year"].to_numpy(dtype=int)
    if len(grp) < 3:
        continue
    for i in range(len(grp) - 2):
        seq = feats[i : i + 2]
        current = seq[-1]
        pooled = seq.mean(axis=0)
        delta = seq[-1] - seq[0]
        short_views.append(current)
        seasonal_views.append(pooled)
        trend_views.append(delta)
        y2.append(np.log1p(np.abs(prems[i + 2])) * np.sign(prems[i + 2]))
        years2.append(yrs[i + 2])

X2_short_raw = np.asarray(short_views, dtype=float)
X2_seasonal_raw = np.asarray(seasonal_views, dtype=float)
X2_trend_raw = np.asarray(trend_views, dtype=float)
y2 = np.asarray(y2, dtype=np.float32)
years2 = np.asarray(years2, dtype=int)
task2_regime_signal = X2_short_raw[:, 0].copy()

split_idx = int(0.8 * len(y2))
X2_train_views_raw = {
    "short": X2_short_raw[:split_idx],
    "seasonal": X2_seasonal_raw[:split_idx],
    "trend": X2_trend_raw[:split_idx],
}
X2_test_views_raw = {
    "short": X2_short_raw[split_idx:],
    "seasonal": X2_seasonal_raw[split_idx:],
    "trend": X2_trend_raw[split_idx:],
}
y2_train = y2[:split_idx]
y2_test = y2[split_idx:]
task2_regime_signal_train = task2_regime_signal[:split_idx]
task2_regime_signal_test = task2_regime_signal[split_idx:]

if TASK2_TRAIN_CAP is not None:
    X2_train_views_raw = {k: v[:TASK2_TRAIN_CAP] for k, v in X2_train_views_raw.items()}
    y2_train = y2_train[:TASK2_TRAIN_CAP]
    task2_regime_signal_train = task2_regime_signal_train[:TASK2_TRAIN_CAP]
if TASK2_TEST_CAP is not None:
    X2_test_views_raw = {k: v[:TASK2_TEST_CAP] for k, v in X2_test_views_raw.items()}
    y2_test = y2_test[:TASK2_TEST_CAP]
    task2_regime_signal_test = task2_regime_signal_test[:TASK2_TEST_CAP]

print("Task 2 raw views:")
for k, v in X2_train_views_raw.items():
    print(f"  {k:<8s} train={v.shape} | test={X2_test_views_raw[k].shape}")
print(f"  regression targets train={len(y2_train)} | test={len(y2_test)}")


---
## 3 — Encode Each View to the Reservoir Width

In [ ]:
def encode_views(train_views_raw, test_views_raw, n_qubits):
    train_encoded, test_encoded, stats = {}, {}, {}
    for name in train_views_raw:
        Xtr = train_views_raw[name]
        Xte = test_views_raw[name]

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)

        n_components = min(n_qubits, Xtr_s.shape[1], max(1, Xtr_s.shape[0] - 1))
        pca = PCA(n_components=n_components, random_state=SEED)
        Xtr_p = pca.fit_transform(Xtr_s)
        Xte_p = pca.transform(Xte_s)

        Xtr_pad = np.zeros((len(Xtr_p), n_qubits), dtype=float)
        Xte_pad = np.zeros((len(Xte_p), n_qubits), dtype=float)
        Xtr_pad[:, :n_components] = Xtr_p
        Xte_pad[:, :n_components] = Xte_p

        mm = MinMaxScaler(feature_range=(0, np.pi))
        Xtr_e = mm.fit_transform(Xtr_pad).astype(np.float32)
        Xte_e = mm.transform(Xte_pad).astype(np.float32)

        train_encoded[name] = Xtr_e
        test_encoded[name] = Xte_e
        stats[name] = {
            "n_components": int(n_components),
            "variance_explained": float(pca.explained_variance_ratio_.sum()),
        }
    return train_encoded, test_encoded, stats

X1_train_views, X1_test_views, enc1_stats = encode_views(X1_train_views_raw, X1_test_views_raw, N_RES_QUBITS)
X2_train_views, X2_test_views, enc2_stats = encode_views(X2_train_views_raw, X2_test_views_raw, N_RES_QUBITS)

print("Task 1A encoding stats:", enc1_stats)
print("Task 2 encoding stats:", enc2_stats)


---
## 4 — Multi-Scale Fixed Quantum Reservoir Bank with Local Projected Observables

In [ ]:
RESERVOIR_CONFIG = [
    {"name": "short", "depth": 4, "pattern": "ring", "seed": SEED + 11},
    {"name": "seasonal", "depth": 6, "pattern": "ladder", "seed": SEED + 23},
    {"name": "trend", "depth": 8, "pattern": "star", "seed": SEED + 37},
]

def apply_entanglement(pattern, n_qubits):
    if pattern == "ring":
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])
    elif pattern == "ladder":
        for i in range(0, n_qubits - 1, 2):
            qml.CNOT(wires=[i, i + 1])
        for i in range(1, n_qubits - 1, 2):
            qml.CNOT(wires=[i, i + 1])
    else:  # star
        for i in range(1, n_qubits):
            qml.CNOT(wires=[0, i])

def make_reservoir_qnode(pattern, n_qubits):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="numpy")
    def circuit(x, params):
        for i in range(n_qubits):
            qml.RY(x[i], wires=i)
            qml.RZ(0.5 * x[(i + 1) % n_qubits], wires=i)

        for layer in range(params.shape[0]):
            for i in range(n_qubits):
                qml.RX(params[layer, i, 0], wires=i)
                qml.RY(params[layer, i, 1], wires=i)
                qml.RZ(params[layer, i, 2], wires=i)
            apply_entanglement(pattern, n_qubits)

        obs = [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
        obs += [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]
        obs += [qml.expval(qml.PauliZ(i) @ qml.PauliZ((i + 1) % n_qubits)) for i in range(n_qubits)]
        return obs

    return circuit

reservoir_params = {}
reservoir_qnodes = {}
for cfg in RESERVOIR_CONFIG:
    rng = np.random.default_rng(cfg["seed"])
    reservoir_params[cfg["name"]] = rng.uniform(0, 2 * np.pi, size=(cfg["depth"], N_RES_QUBITS, 3))
    reservoir_qnodes[cfg["name"]] = make_reservoir_qnode(cfg["pattern"], N_RES_QUBITS)

def extract_reservoir_bank_features(view_dict, label=""):
    blocks = []
    for cfg in RESERVOIR_CONFIG:
        name = cfg["name"]
        X = view_dict[name]
        qnode = reservoir_qnodes[name]
        params = reservoir_params[name]
        feats = []
        t0 = time.time()
        for idx, x in enumerate(X):
            feats.append(qnode(x, params))
            if (idx + 1) % 100 == 0:
                print(f"  {label} {name:<8s} {idx+1}/{len(X)}  ({time.time()-t0:.1f}s)", end="\r")
        block = np.asarray(feats, dtype=np.float32)
        print(f"  {label} {name:<8s} {len(X)}/{len(X)} done in {time.time()-t0:.1f}s")
        blocks.append(block)
    return np.concatenate(blocks, axis=1)

print("Reservoir bank ready.")
print(f"Feature width after bank = {len(RESERVOIR_CONFIG) * RESERVOIR_OBS_PER_BANK}")


---
## 5 — Extract Reservoir Features for Both Tasks

In [ ]:
print("Extracting Task 1A reservoir features...")
X1_res_train = extract_reservoir_bank_features(X1_train_views, label="Task1 train")
X1_res_test = extract_reservoir_bank_features(X1_test_views, label="Task1 test ")

print("\nExtracting Task 2 reservoir features...")
X2_res_train = extract_reservoir_bank_features(X2_train_views, label="Task2 train")
X2_res_test = extract_reservoir_bank_features(X2_test_views, label="Task2 test ")

print("\nReservoir feature shapes:")
print("  Task 1A:", X1_res_train.shape, X1_res_test.shape)
print("  Task 2 :", X2_res_train.shape, X2_res_test.shape)


---
## 6 — Baseline Ablation: Linear Readout on Multi-Scale Reservoir Features

In [ ]:
clf_lin = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
clf_lin.fit(X1_res_train, y1_train)
y1_lin_prob_train = clf_lin.predict_proba(X1_res_train)[:, 1]
y1_lin_pred = clf_lin.predict(X1_res_test)
y1_lin_prob = clf_lin.predict_proba(X1_res_test)[:, 1]

task1_linear = {
    "f1": float(f1_score(y1_test, y1_lin_pred)),
    "auc": float(roc_auc_score(y1_test, y1_lin_prob)),
}

reg_lin = Ridge(alpha=1.0)
reg_lin.fit(X2_res_train, y2_train)
y2_lin_pred_train = reg_lin.predict(X2_res_train)
y2_lin_pred = reg_lin.predict(X2_res_test)

task2_linear = {
    "r2": float(r2_score(y2_test, y2_lin_pred)),
    "rmse": float(np.sqrt(mean_squared_error(y2_test, y2_lin_pred))),
}

def make_quantile_regimes(train_signal, test_signal, n_bins=N_REGIMES):
    edges = np.quantile(train_signal, np.linspace(0, 1, n_bins + 1))
    edges[0] = -np.inf
    edges[-1] = np.inf
    for i in range(1, len(edges) - 1):
        if edges[i] <= edges[i - 1]:
            edges[i] = edges[i - 1] + 1e-6
    train_reg = np.digitize(train_signal, edges[1:-1], right=False)
    test_reg = np.digitize(test_signal, edges[1:-1], right=False)
    return train_reg.astype(int), test_reg.astype(int), edges

def make_feature_space_regimes(train_features, test_features, n_bins=N_REGIMES):
    return make_quantile_regimes(train_features.reshape(-1), test_features.reshape(-1), n_bins=n_bins)

task1_reg_train, task1_reg_test, task1_reg_edges = make_quantile_regimes(
    task1_regime_signal_train, task1_regime_signal_test, n_bins=N_REGIMES
)
task2_reg_train, task2_reg_test, task2_reg_edges = make_quantile_regimes(
    task2_regime_signal_train, task2_regime_signal_test, n_bins=N_REGIMES
)

print("=== Multi-scale Reservoir + Linear Head ===")
print(f"Task 1A  F1={task1_linear['f1']:.4f} | AUC={task1_linear['auc']:.4f}")
print(f"Task 2   R2={task2_linear['r2']:.4f} | RMSE={task2_linear['rmse']:.4f}")
print("\nTask 1A regime counts:", np.bincount(task1_reg_train, minlength=N_REGIMES))
print("Task 2 regime counts:", np.bincount(task2_reg_train, minlength=N_REGIMES))


---
## 7 — Regime-Routed Residual Re-uploading Experts

This second-pass head is more conservative than the first attempt.

Implementation choices:

- start from the **linear baseline**
- train the quantum block as a **small residual correction**
- route samples through **quantile-based regime experts**
- keep corrections small with a bounded residual scale
- train layerwise: 1 layer first, then 2 layers


In [ ]:
def safe_logit(p, eps=1e-4):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def best_f1_threshold(y_true, probs):
    grid = np.linspace(0.25, 0.75, 41)
    scores = [f1_score(y_true, (probs >= t).astype(int)) for t in grid]
    return float(grid[int(np.argmax(scores))])

def make_reupload_qnode(n_layers, n_qubits):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def circuit(inputs, weights):
        for layer in range(n_layers):
            for i in range(n_qubits):
                qml.RY(inputs[i], wires=i)
                qml.RZ(0.5 * inputs[(i + 1) % n_qubits], wires=i)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])
            qml.CNOT(wires=[n_qubits - 1, 0])
            for i in range(n_qubits):
                qml.RX(weights[layer, i, 0], wires=i)
                qml.RY(weights[layer, i, 1], wires=i)
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circuit

class ResidualExpertHead(nn.Module):
    def __init__(self, input_dim, n_layers=1, n_qubits=N_RES_QUBITS, correction_cap=0.10):
        super().__init__()
        self.n_layers = n_layers
        self.n_qubits = n_qubits
        self.correction_cap = correction_cap

        self.pre = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Linear(20, n_qubits),
        )
        self.scale = nn.Parameter(torch.tensor(np.pi / 2, dtype=torch.float32))
        weight_shapes = {"weights": (n_layers, n_qubits, 2)}
        self.quantum = qml.qnn.TorchLayer(make_reupload_qnode(n_layers, n_qubits), weight_shapes)
        self.post = nn.Linear(n_qubits, 1)
        self.correction_gate = nn.Parameter(torch.tensor(0.0, dtype=torch.float32))
        self.reset_parameters()

    def reset_parameters(self):
        for module in self.pre:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
        for _, param in self.quantum.named_parameters():
            nn.init.zeros_(param)
        nn.init.normal_(self.post.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.post.bias)

    def forward(self, x):
        h = self.pre(x)
        q_in = torch.tanh(h) * self.scale
        if q_in.ndim == 1:
            q_out = self.quantum(q_in)
        else:
            q_out = torch.stack([self.quantum(sample) for sample in q_in], dim=0)
        out = self.post(q_out).squeeze(-1)
        residual_scale = self.correction_cap * torch.sigmoid(self.correction_gate)
        return out * residual_scale

def warm_start_from_previous(prev_model, new_model):
    new_model.pre.load_state_dict(prev_model.pre.state_dict())
    new_model.post.load_state_dict(prev_model.post.state_dict())
    new_model.scale.data = prev_model.scale.data.clone()
    new_model.correction_gate.data = prev_model.correction_gate.data.clone()

    prev_state = prev_model.quantum.state_dict()
    new_state = new_model.quantum.state_dict()
    for key, old_val in prev_state.items():
        if key not in new_state:
            continue
        new_val = new_state[key]
        if old_val.shape == new_val.shape:
            new_state[key] = old_val.clone()
        elif old_val.ndim >= 1 and old_val.shape[0] < new_val.shape[0]:
            patched = new_val.clone()
            patched[: old_val.shape[0]] = old_val
            new_state[key] = patched
    new_model.quantum.load_state_dict(new_state)

def train_single_residual_expert(
    X_train,
    y_train,
    baseline_train,
    X_test,
    baseline_test,
    task="classification",
    layer_schedule=HEAD_LAYER_SCHEDULE,
    epochs_per_stage=6,
    batch_size=24,
    lr=0.006,
    correction_lambda=0.01,
    correction_cap=0.10,
    residual_bias=0.0,
):
    Xtr_t = torch.tensor(X_train, dtype=torch.float32)
    ytr_t = torch.tensor(y_train, dtype=torch.float32)
    base_tr_t = torch.tensor(baseline_train, dtype=torch.float32)
    Xte_t = torch.tensor(X_test, dtype=torch.float32) if len(X_test) else None
    base_te_t = torch.tensor(baseline_test, dtype=torch.float32) if len(baseline_test) else None

    dataset = TensorDataset(Xtr_t, ytr_t, base_tr_t)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    prev_model = None
    history = []
    residual_bias = float(residual_bias)

    for n_layers in layer_schedule:
        model = ResidualExpertHead(X_train.shape[1], n_layers=n_layers, correction_cap=correction_cap)
        if prev_model is not None:
            warm_start_from_previous(prev_model, model)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        if task == "classification":
            pos = max(float(y_train.sum()), 1.0)
            neg = max(float(len(y_train) - y_train.sum()), 1.0)
            pos_weight = torch.tensor(neg / pos, dtype=torch.float32)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            criterion = nn.SmoothL1Loss(beta=0.5)
            target_residual = torch.tensor(y_train - baseline_train - residual_bias, dtype=torch.float32)
            dataset = TensorDataset(Xtr_t, ytr_t, base_tr_t, target_residual)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        t0 = time.time()
        for epoch in range(epochs_per_stage):
            model.train()
            epoch_loss = 0.0
            for batch in loader:
                optimizer.zero_grad()
                if task == "classification":
                    xb, yb, bb = batch
                else:
                    xb, yb, bb, rb = batch
                corr = model(xb)
                if task == "classification":
                    pred = bb + corr
                    loss = criterion(pred, yb)
                else:
                    loss = criterion(corr, rb)
                loss = loss + correction_lambda * (corr ** 2).mean()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            if (epoch + 1) % max(1, epochs_per_stage // 2) == 0:
                print(
                    f"  layers={n_layers} epoch={epoch+1:02d} "
                    f"loss={epoch_loss/len(loader):.4f} "
                    f"time={time.time()-t0:.1f}s"
                )
        history.append({"layers": int(n_layers), "final_loss": float(epoch_loss / len(loader))})
        prev_model = model

    prev_model.eval()
    with torch.no_grad():
        corr_train = prev_model(Xtr_t).cpu().numpy().astype(float)
        if len(X_test):
            corr_test = prev_model(Xte_t).cpu().numpy().astype(float)
        else:
            corr_test = np.array([], dtype=float)

    if task == "classification":
        pred_train = 1.0 / (1.0 + np.exp(-(baseline_train + corr_train)))
        pred_test = 1.0 / (1.0 + np.exp(-(baseline_test + corr_test))) if len(corr_test) else corr_test
    else:
        pred_train = baseline_train + residual_bias + corr_train
        pred_test = baseline_test + residual_bias + corr_test if len(corr_test) else corr_test

    return prev_model, pred_train, pred_test, history

def train_routed_residual_experts(
    X_train,
    y_train,
    baseline_train,
    regime_train,
    X_test,
    y_test,
    baseline_test,
    regime_test,
    task="classification",
    layer_schedule=HEAD_LAYER_SCHEDULE,
    epochs_per_stage=6,
    batch_size=24,
    lr=0.006,
    min_samples=30,
):
    final_train = baseline_train.astype(float).copy()
    final_test = baseline_test.astype(float).copy()
    expert_summaries = []

    for regime in range(N_REGIMES):
        tr_idx = np.where(regime_train == regime)[0]
        te_idx = np.where(regime_test == regime)[0]
        if len(tr_idx) < min_samples:
            expert_summaries.append(
                {"regime": int(regime), "train_samples": int(len(tr_idx)), "test_samples": int(len(te_idx)), "status": "skipped"}
            )
            continue

        if task == "classification":
            base_tr = safe_logit(baseline_train[tr_idx])
            base_te = safe_logit(baseline_test[te_idx]) if len(te_idx) else np.array([], dtype=float)
            residual_bias = 0.0
        else:
            base_tr = baseline_train[tr_idx]
            base_te = baseline_test[te_idx] if len(te_idx) else np.array([], dtype=float)
            residual_bias = float((y_train[tr_idx] - base_tr).mean())

        model, pred_tr, pred_te, hist = train_single_residual_expert(
            X_train[tr_idx],
            y_train[tr_idx],
            base_tr,
            X_test[te_idx] if len(te_idx) else np.empty((0, X_test.shape[1]), dtype=np.float32),
            base_te,
            task=task,
            layer_schedule=layer_schedule,
            epochs_per_stage=epochs_per_stage,
            batch_size=batch_size,
            lr=lr,
            correction_lambda=0.02 if task == "classification" else 0.01,
            correction_cap=0.10 if task == "classification" else 0.05,
            residual_bias=residual_bias,
        )

        final_train[tr_idx] = pred_tr
        if len(te_idx):
            final_test[te_idx] = pred_te

        expert_summaries.append(
            {
                "regime": int(regime),
                "train_samples": int(len(tr_idx)),
                "test_samples": int(len(te_idx)),
                "status": "trained",
                "residual_bias": residual_bias,
                "history": hist,
            }
        )

    if task == "classification":
        threshold = best_f1_threshold(y_train, final_train)
        pred_label = (final_test >= threshold).astype(int)
        metrics = {
            "f1": float(f1_score(y_test, pred_label)),
            "auc": float(roc_auc_score(y_test, final_test)),
            "threshold": float(threshold),
        }
    else:
        metrics = {
            "r2": float(r2_score(y_test, final_test)),
            "rmse": float(np.sqrt(mean_squared_error(y_test, final_test))),
        }

    return final_train, final_test, metrics, expert_summaries


---
## 8 — Train Regime-Routed Residual RAM-QRSRE on Task 1A and Task 2

In [ ]:
print("Training Task 1A routed residual experts...")
y1_q_train, y1_q_prob, task1_qhead, hist1 = train_routed_residual_experts(
    X1_res_train,
    y1_train.astype(np.float32),
    y1_lin_prob_train,
    task1_reg_train,
    X1_res_test,
    y1_test.astype(np.float32),
    y1_lin_prob,
    task1_reg_test,
    task="classification",
    epochs_per_stage=6 if not CLOUD_MODE else 18,
    batch_size=24,
    lr=0.006,
    min_samples=40,
)
y1_q_pred = (y1_q_prob >= task1_qhead["threshold"]).astype(int)

print("\nTraining Task 2 routed residual experts...")
y2_q_train, y2_q_pred, task2_qhead, hist2 = train_routed_residual_experts(
    X2_res_train,
    y2_train.astype(np.float32),
    y2_lin_pred_train,
    task2_reg_train,
    X2_res_test,
    y2_test.astype(np.float32),
    y2_lin_pred,
    task2_reg_test,
    task="regression",
    epochs_per_stage=10 if not CLOUD_MODE else 22,
    batch_size=24,
    lr=0.004,
    min_samples=25,
)

print("\n=== REGIME-ROUTED RESIDUAL RAM-QRSRE RESULTS ===")
print(f"Task 1A  F1={task1_qhead['f1']:.4f} | AUC={task1_qhead['auc']:.4f} | threshold={task1_qhead['threshold']:.3f}")
print(f"Task 2   R2={task2_qhead['r2']:.4f} | RMSE={task2_qhead['rmse']:.4f}")
print("\nTask 1A classification report:")
print(classification_report(y1_test, y1_q_pred, target_names=['No Fire', 'Fire']))


---
## 9 — Comparison, Resource Table, and Save Results

In [ ]:
reference_repo_best = {
    "task1_qrc_f1": 0.7241,
    "task2_qlstm_r2": 0.9221,
}

comparison = pd.DataFrame(
    [
        {"Task": "1A", "Model": "Multi-scale Reservoir + Linear Head", "Metric": "F1", "Score": task1_linear["f1"], "Type": "Hybrid"},
        {"Task": "1A", "Model": "Routed Residual RAM-QRSRE", "Metric": "F1", "Score": task1_qhead["f1"], "Type": "Quantum-Hybrid"},
        {"Task": "1A", "Model": "Repo reference: QRC + LogReg", "Metric": "F1", "Score": reference_repo_best["task1_qrc_f1"], "Type": "Reference"},
        {"Task": "2", "Model": "Multi-scale Reservoir + Ridge", "Metric": "R2", "Score": task2_linear["r2"], "Type": "Hybrid"},
        {"Task": "2", "Model": "Routed Residual RAM-QRSRE", "Metric": "R2", "Score": task2_qhead["r2"], "Type": "Quantum-Hybrid"},
        {"Task": "2", "Model": "Repo reference: QLSTM", "Metric": "R2", "Score": reference_repo_best["task2_qlstm_r2"], "Type": "Reference"},
    ]
)

ablations = pd.DataFrame(
    [
        {"Task": "1A", "Variant": "Reservoir + Linear Head", "PrimaryMetric": task1_linear["f1"], "AuxMetric": task1_linear["auc"]},
        {"Task": "1A", "Variant": "Routed Residual RAM-QRSRE", "PrimaryMetric": task1_qhead["f1"], "AuxMetric": task1_qhead["auc"]},
        {"Task": "2", "Variant": "Reservoir + Linear Head", "PrimaryMetric": task2_linear["r2"], "AuxMetric": task2_linear["rmse"]},
        {"Task": "2", "Variant": "Routed Residual RAM-QRSRE", "PrimaryMetric": task2_qhead["r2"], "AuxMetric": task2_qhead["rmse"]},
    ]
)

resource_table = pd.DataFrame(
    [
        {
            "Component": "Reservoir bank",
            "Qubits": N_RES_QUBITS,
            "Count": len(RESERVOIR_CONFIG),
            "Depths": ",".join(str(cfg["depth"]) for cfg in RESERVOIR_CONFIG),
            "TrainableParams": 0,
        },
        {
            "Component": "Residual re-uploading experts",
            "Qubits": N_RES_QUBITS,
            "Count": N_REGIMES,
            "Depths": max(HEAD_LAYER_SCHEDULE),
            "TrainableParams": N_REGIMES * (max(HEAD_LAYER_SCHEDULE) * N_RES_QUBITS * 2),
        },
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

task1_plot = comparison[comparison["Task"] == "1A"]
axes[0].bar(task1_plot["Model"], task1_plot["Score"], color=["#78909C", "#1565C0", "#D32F2F"])
axes[0].set_title("Task 1A — F1 Comparison")
axes[0].set_ylabel("F1")
axes[0].tick_params(axis="x", rotation=20)

task2_plot = comparison[comparison["Task"] == "2"]
axes[1].bar(task2_plot["Model"], task2_plot["Score"], color=["#78909C", "#1565C0", "#D32F2F"])
axes[1].set_title("Task 2 — R2 Comparison")
axes[1].set_ylabel("R2")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(f"../results/{RESULT_PREFIX}_comparison.png", dpi=150, bbox_inches="tight")
if IN_NOTEBOOK:
    plt.show()
else:
    plt.close(fig)

results_payload = {
    "task_1a": {
        "linear_head": task1_linear,
        "ram_qrsre": task1_qhead,
        "expert_history": hist1,
        "regime_edges": task1_reg_edges.tolist(),
    },
    "task_2": {
        "linear_head": task2_linear,
        "ram_qrsre": task2_qhead,
        "expert_history": hist2,
        "regime_edges": task2_reg_edges.tolist(),
    },
    "config": {
        "reservoir_qubits": N_RES_QUBITS,
        "reservoirs": RESERVOIR_CONFIG,
        "head_layer_schedule": list(HEAD_LAYER_SCHEDULE),
        "projected_local_observables_per_reservoir": RESERVOIR_OBS_PER_BANK,
        "n_regimes": N_REGIMES,
    },
}

with open(f"../results/{RESULT_PREFIX}_results.json", "w") as f:
    _json.dump(results_payload, f, indent=2)

ablations.to_csv(f"../results/{RESULT_PREFIX}_ablation.csv", index=False)
resource_table.to_csv(f"../results/{RESULT_PREFIX}_resource_table.csv", index=False)

print("Saved:")
print(f"  ../results/{RESULT_PREFIX}_results.json")
print(f"  ../results/{RESULT_PREFIX}_ablation.csv")
print(f"  ../results/{RESULT_PREFIX}_resource_table.csv")
print(f"  ../results/{RESULT_PREFIX}_comparison.png")


---
## 10 — Interpretation

What this notebook tells us:

- whether **multi-scale fixed reservoirs** are already useful without a trainable quantum head,
- whether a **regime-routed residual quantum correction** adds value over a linear readout,
- and how far this architecture gets relative to the repo's current best QRC and QLSTM reference scores.

If this second-pass routed head helps, the next step is:

1. test **learned routing** instead of quantile routing,
2. test **shared multitask backbones**,
3. widen the ablations,
4. optionally try **quantum natural gradient** for the head.
